In [ ]:
import pandas as pd
from datasets import Dataset

# Load your dataset
df = pd.read_csv("Cleaned_Indian_Food_Dataset.csv")

# Create a prompt template
# We combine Ingredients AND Cuisine into the input so the model learns to respect both.
def format_instruction(row):
    return {
        "instruction": f"I have these ingredients: {row['Cleaned-Ingredients']}. I want a {row['Cuisine']} dish.",
        "input": "",
        "output": f"Here is a recipe for {row['TranslatedRecipeName']}.\n\nIngredients:\n{row['TranslatedIngredients']}\n\nInstructions:\n{row['TranslatedInstructions']}"
    }

# Convert to list of dicts
data = df.apply(format_instruction, axis=1).tolist()

# Create a HuggingFace dataset
dataset = Dataset.from_list(data)

# Save to JSONL (standard format for LLM training)
dataset.to_json("indian_food_training_data.jsonl")

Creating json from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

11368629

In [ ]:
%%capture
# Installs Unsloth, Xformers, and TRL
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load the 4-bit Quantized Model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Load your dataset
dataset = load_dataset("json", data_files="indian_food_training_data.jsonl", split="train")

# 4. Format Prompts
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. Train
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Set to 300 for better results
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/unsloth/models/rl_replacements.py:946: UserWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  import trl.experimental.openenv.utils as openenv_utils


==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.000500
2,1.858000
3,1.869500
4,1.846000
5,1.768000
6,1.581800
7,1.622300
8,1.529000
9,1.525000
10,1.443200


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
train/grad_norm,▂▃▂▂▁▆▅█▄▃▂▂▃▃▂▂▃▂▁▂▂▂▂▂▅▂▂▃▁▂▂▁▃▄▂▂▂▁▂▃
train/learning_rate,▂▄▅▇█▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,█▇▇▇▆▅▄▄▄▄▄▃▃▃▂▂▃▂▃▃▂▂▂▃▄▁▃▂▂▂▃▂▂▁▃▂▁▁▂▃
total_flos,1.2532751147139072e+16
train/epoch,0.08084
train/global_step,60
train/grad_norm,0.36868
train/learning_rate,0.0
train/loss,1.3593


TrainOutput(global_step=60, training_loss=1.3805952986081442, metrics={'train_runtime': 1021.2393, 'train_samples_per_second': 0.47, 'train_steps_per_second': 0.059, 'total_flos': 1.2532751147139072e+16, 'train_loss': 1.3805952986081442, 'epoch': 0.08083529808016167})

In [ ]:
# Save the fine-tuned model
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("✅ Model saved successfully to 'lora_model' folder.")

✅ Model saved successfully to 'lora_model' folder.


In [ ]:
import os
if os.path.exists("lora_model"):
    print("✅ Success! Model found. You can run the App now.")
    print("Files:", os.listdir("lora_model"))
else:
    print("❌ Error: 'lora_model' folder is missing.")
    print("👉 Solution: Run the TRAINING cell again and wait for it to finish.")

✅ Success! Model found. You can run the App now.
Files: ['tokenizer_config.json', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']


In [ ]:
from unsloth import FastLanguageModel

# Load your fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # The folder where you saved the model
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

def get_recommendation():
    print("🤖: Hi! I am your Indian Food Chef. Tell me, what ingredients do you have? (e.g., Chicken, Rice, Curd)")
    user_ingredients = input("You: ")

    print("\n🤖: Got it. Do you have a specific cuisine preference? (e.g., North Indian, South Indian, Bengali, or 'Any')")
    user_cuisine = input("You: ")

    # Construct the prompt exactly how we trained it
    if user_cuisine.lower() == "any":
        user_cuisine = "Indian" # Default fallback

    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I have these ingredients: {user_ingredients}. I want a {user_cuisine} dish.

### Response:
"""

    # Generate
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)

    # Decode only the new part
    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    print("\n" + response.split("### Response:")[-1])

# Run the app
get_recommendation()

==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🤖: Hi! I am your Indian Food Chef. Tell me, what ingredients do you have? (e.g., Chicken, Rice, Curd)
You: Chicken, Cheese, Potato

🤖: Got it. Do you have a specific cuisine preference? (e.g., North Indian, South Indian, Bengali, or 'Any')
You: North Indian


Here is a recipe for Cheese Potato And Chicken Recipe.

Ingredients:
1/2 teaspoon Turmeric powder (Haldi),1/2 teaspoon Red Chilli powder,1 tablespoon Ginger Garlic paste,1 teaspoon Cumin seeds (Jeera),1/2 cup Fresh cream,1 tablespoon Ghee,1/2 teaspoon Salt,1/2 cup Tomato puree,1/2 cup Milk,

In [ ]:
import re

# Enable inference mode
FastLanguageModel.for_inference(model)

def smart_indian_chef():
    print("="*70)
    print("🍛 AI SMART CHEF: Tell me what you are craving! 🍛")
    print("(e.g., 'I want a quick and easy north indian paneer dish')")
    print("="*70)

    # 1. Get the natural language prompt
    user_query = input("👉 YOU: ").lower()

    # 2. Python Search Logic (The "Router")
    # We scan your dataframe (df) to find recipes that match the user's words.

    # -- Filter by Ingredients (e.g., "paneer", "chicken") --
    # We look for words in the query that appear in the 'Cleaned-Ingredients' column
    matched_recipes = df.copy()

    # -- Filter by Cuisine (North/South/etc) --
    if "north indian" in user_query:
        matched_recipes = matched_recipes[matched_recipes['Cuisine'].str.contains("North Indian", case=False)]
    elif "south indian" in user_query:
        matched_recipes = matched_recipes[matched_recipes['Cuisine'].str.contains("South Indian", case=False)]
    elif "bengali" in user_query:
        matched_recipes = matched_recipes[matched_recipes['Cuisine'].str.contains("Bengali", case=False)]

    # -- Filter by "Quick" (Time < 30 mins) --
    if "quick" in user_query or "fast" in user_query or "easy" in user_query:
        matched_recipes = matched_recipes[matched_recipes['TotalTimeInMins'] <= 30]

    # -- Filter by Ingredient Keywords in query --
    # Simple check: if the user said "paneer", we only keep rows with "paneer"
    common_ingredients = ["paneer", "chicken", "potato", "aloo", "rice", "dal", "spinach", "mutton", "fish", "egg"]
    for ing in common_ingredients:
        if ing in user_query:
            matched_recipes = matched_recipes[matched_recipes['Cleaned-Ingredients'].str.contains(ing, case=False)]

    # 3. Show Recommendations
    if matched_recipes.empty:
        print("\n🤖 CHEF: Hmm, I couldn't find a perfect match. Try listing ingredients simply? (e.g., 'Potato, Spinach')")
        return

    # Get top 5 random results to show variety
    recommendations = matched_recipes.head(5)

    print(f"\n🤖 CHEF: I found {len(matched_recipes)} recipes! Here are the top 5 matches:")
    options = []
    for idx, (i, row) in enumerate(recommendations.iterrows()):
        print(f"   {idx+1}. {row['TranslatedRecipeName']} ({row['TotalTimeInMins']} mins)")
        options.append(row)

    # 4. User Selection
    print("\n👉 Select a number (1-5) to generate the step-by-step guide:")
    try:
        choice = int(input("YOU: ")) - 1
        selected_dish = options[choice]
    except:
        print("Invalid choice!")
        return

    # 5. Generate the Full Guide using the LLM
    # We feed the ACTUAL ingredients of the selected dish to the LLM to get the generation.
    print(f"\n👨‍🍳 CHEF: Excellent choice! Generating the guide for {selected_dish['TranslatedRecipeName']}...\n")

    prompt_input = f"I have these ingredients: {selected_dish['Cleaned-Ingredients']}. I want a {selected_dish['Cuisine']} dish."

    alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
"""
    formatted_prompt = alpaca_prompt.format(prompt_input)

    inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.6, # Slightly higher creativity for the instructions
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    final_recipe = response.split("### Response:")[-1].strip()

    print("-" * 60)
    print(final_recipe)
    print("-" * 60)

# Run the smart chef
smart_indian_chef()

🍛 AI SMART CHEF: Tell me what you are craving! 🍛
(e.g., 'I want a quick and easy north indian paneer dish')
👉 YOU: I want a easy dish with chicken

🤖 CHEF: I found 49 recipes! Here are the top 5 matches:
   1. Chicken Tikka Taco Recipe Topped With Cheesy Garlic Mayo (20 mins)
   2. PF Chang's Style Crispy Chicken Lettuce Wraps Recipe (25 mins)
   3. Chicken With Mushroom Sauce Recipe (25 mins)
   4. Grilled Chicken With Vegetables Recipe (25 mins)
   5. Garlic Red Chicken Gravy Recipe (20 mins)

👉 Select a number (1-5) to generate the step-by-step guide:
YOU: 3

👨‍🍳 CHEF: Excellent choice! Generating the guide for Chicken With Mushroom Sauce Recipe...

------------------------------------------------------------
Here is a recipe for Chicken Mushroom Steak Recipe.

Ingredients:
2 cloves Garlic - minced,1 teaspoon Thyme leaves - chopped,1 teaspoon Red Chilli flakes,1 Onion - finely chopped,1 teaspoon Black pepper powder,Salt - to taste,1/2 cup Corn flour,1/2 cup Milk,1/2 cup Butter - mel

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from unsloth import FastLanguageModel
import pandas as pd

# 1. Load the embedding model (If you already ran this line, you can skip it)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Pre-compute embeddings (Only if not done yet)
if 'semantic_text' not in df.columns:
    print("⏳ Creating semantic index... (this takes ~30 seconds)")
    df['semantic_text'] = df['TranslatedRecipeName'] + " " + df['Cleaned-Ingredients'] + " " + df['Cuisine']
    recipe_embeddings = embedder.encode(df['semantic_text'].tolist(), show_progress_bar=True)

# Enable inference
FastLanguageModel.for_inference(model)

def semantic_chef():
    print("="*70)
    print("🧠 SEMANTIC AI CHEF: Ask me anything (e.g., 'Something for a rainy day')")
    print("="*70)

    user_query = input("👉 YOU: ")

    # 1. Convert user query to vector
    query_embedding = embedder.encode([user_query])

    # 2. Calculate Similarity
    # We use util.cos_sim or just the model's similarity method if available
    similarities = embedder.similarity(query_embedding, recipe_embeddings)[0]

    # 3. Get Top 5 matches
    # FIX: We add .copy() here to fix the "negative stride" error
    top_k_indices = np.argsort(similarities.numpy())[-5:][::-1].copy()

    recommendations = df.iloc[top_k_indices]

    # We also need to copy the scores to print them safely
    scores = similarities[top_k_indices].numpy()

    print(f"\n🤖 CHEF: meaningful matches found! (Scores: {scores})")
    options = []
    for idx, (i, row) in enumerate(recommendations.iterrows()):
        print(f"   {idx+1}. {row['TranslatedRecipeName']} ({row['Cuisine']})")
        options.append(row)

    # 4. User Selection
    print("\n👉 Select a number (1-5) to generate the guide:")
    try:
        selection = input("YOU: ")
        if not selection.isdigit():
            print("❌ Please enter a number.")
            return
        choice = int(selection) - 1
        selected_dish = options[choice]
    except (IndexError, ValueError):
        print("❌ Invalid selection.")
        return

    # 5. Generation
    print(f"\n👨‍🍳 CHEF: Generating guide for {selected_dish['TranslatedRecipeName']}...\n")

    # We feed the EXACT ingredients from the database to the LLM
    prompt_input = f"I have these ingredients: {selected_dish['Cleaned-Ingredients']}. I want a {selected_dish['Cuisine']} dish."

    alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
"""
    formatted_prompt = alpaca_prompt.format(prompt_input)
    inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.6,
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    print(response.split("### Response:")[-1].strip())

# Run the fixed chef
semantic_chef()

🧠 SEMANTIC AI CHEF: Ask me anything (e.g., 'Something for a rainy day')
👉 YOU: A easy dish with chicken

🤖 CHEF: meaningful matches found! (Scores: [0.6051172  0.60260236 0.59252304 0.5903505  0.5892408 ])
   1. Chicken In Lemon Butter Sauce Recipe (Continental)
   2. Simple & Easy Chicken Cutlet Recipe - High Protein Dish (Continental)
   3. Grilled Chicken With Vegetables Recipe (Indian)
   4. Restaurant Style Chicken 65 Recipe (South Indian Recipes)
   5. Easy Creamy Chicken Curry Recipe (North Indian Recipes)

👉 Select a number (1-5) to generate the guide:
YOU: 5

👨‍🍳 CHEF: Generating guide for Easy Creamy Chicken Curry Recipe...

Here is a recipe for Chicken Tikka Masala Recipe.

Ingredients:
1/2 teaspoon Black pepper powder,1 teaspoon Sunflower Oil,Salt - to taste,1/2 cup Curd (Dahi / Yogurt),1 teaspoon Lemon juice,1/2 teaspoon Kasuri Methi (Dried Fenugreek Leaves),2 inch Ginger - grated,1/2 cup Chicken Breast - cut into 1 inch cubes,2 cloves Garlic - grated,1/2 teaspoon Kasuri m

In [ ]:
# 1. Install the embedding library (Run this once)
!pip install sentence-transformers

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel

# --- SETUP: Load Models ---

# A. Load the Semantic Bridge (Embedding Model)
# This model converts text into numbers (vectors) to find meaning.
print("⏳ Loading Semantic Model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# B. Load your Fine-Tuned Llama 3 (if not already loaded)
# model, tokenizer = FastLanguageModel.from_pretrained("lora_model", load_in_4bit=True)
FastLanguageModel.for_inference(model)

# --- STEP 1: Index the Data (The Bridge) ---
# We create a "Meaning" column that combines Name, Ingredients, and Cuisine
if 'semantic_text' not in df.columns:
    print("⏳ Creating semantic index (this takes ~30 seconds)...")
    df['semantic_text'] = df['TranslatedRecipeName'] + " " + df['Cleaned-Ingredients'] + " " + df['Cuisine']

    # Convert all recipes to vectors
    recipe_embeddings = embedder.encode(df['semantic_text'].tolist(), show_progress_bar=True)
else:
    print("✅ Semantic index already exists!")

# --- STEP 2: The Semantic Application ---

def semantic_chef_app():
    print("\n" + "="*60)
    print("🧠 SEMANTIC AI CHEF: I understand intent, not just keywords.")
    print("   Try: 'Something for a rainy day', 'High protein vegetarian', or 'Sweet dessert for kids'")
    print("="*60)

    # 1. Get User Intent
    user_query = input("👉 YOU: ")

    # 2. Semantic Search (The Bridge)
    # Convert query to vector -> Find closest recipe vectors
    query_embedding = embedder.encode([user_query])
    similarities = embedder.similarity(query_embedding, recipe_embeddings)[0]

    # Get Top 5 matches (Fixing the negative stride error with .copy())
    top_indices = np.argsort(similarities.numpy())[-5:][::-1].copy()
    top_scores = similarities[top_indices].numpy()

    # 3. Present Options
    print(f"\n🤖 CHEF: I found these dishes matching '{user_query}':")
    recommendations = []
    for idx, i in enumerate(top_indices):
        row = df.iloc[i]
        print(f"   {idx+1}. {row['TranslatedRecipeName']} (Match: {top_scores[idx]:.2f})")
        recommendations.append(row)

    # 4. User Selection
    print("\n👉 Select a number (1-5) to generate the cooking guide:")
    try:
        choice = int(input("YOU: ")) - 1
        selected_dish = recommendations[choice]
    except:
        print("❌ Invalid selection.")
        return

    # 5. Generative Response (Llama 3)
    # We feed the EXACT facts to Llama 3 so it doesn't hallucinate
    print(f"\n👨‍🍳 CHEF: Perfect choice! Writing the guide for {selected_dish['TranslatedRecipeName']}...\n")

    prompt_input = f"I have these ingredients: {selected_dish['Cleaned-Ingredients']}. I want a {selected_dish['Cuisine']} dish."

    alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
"""
    formatted_prompt = alpaca_prompt.format(prompt_input)

    inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.6,
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    final_output = response.split("### Response:")[-1].strip()

    print("-" * 60)
    print(final_output)
    print("-" * 60)

# Run the app
semantic_chef_app()

⏳ Loading Semantic Model...
✅ Semantic index already exists!

🧠 SEMANTIC AI CHEF: I understand intent, not just keywords.
   Try: 'Something for a rainy day', 'High protein vegetarian', or 'Sweet dessert for kids'
👉 YOU: Low calorie high protein dish

🤖 CHEF: I found these dishes matching 'Low calorie high protein dish':
   1. High Protein Chickpea Potato Hash Brown Recipe (Match: 0.48)
   2. High Protein Spinach & Soy Bites Recipe (Match: 0.47)
   3. Spirulina Protein Energy Balls with Ragi Recipe (Match: 0.46)
   4. High Protein Soya Dosa Recipe (Match: 0.45)
   5. High Protein Grilled Rajma Corn Sandwich Recipe (Match: 0.45)

👉 Select a number (1-5) to generate the cooking guide:
YOU: 4

👨‍🍳 CHEF: Perfect choice! Writing the guide for High Protein Soya Dosa Recipe...

------------------------------------------------------------
Here is a recipe for South Indian Style Idli Recipe.

Ingredients:
1 cup Idli Rice,2 tablespoons Chana Dal (Bengal Gram Dal),1 cup White Urad Dal,1/4 cup Soy

# Training loss curve (The Training chart)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract logs from the trainer
# Note: This requires the 'trainer' object from the training step.
if 'trainer' in locals() and trainer.state.log_history:
    history = trainer.state.log_history

    # Filter for training loss
    loss_data = [x for x in history if 'loss' in x]
    steps = [x['step'] for x in loss_data]
    losses = [x['loss'] for x in loss_data]

    plt.figure(figsize=(10, 5))
    sns.lineplot(x=steps, y=losses, marker='o', color='#ff6b6b', linewidth=2.5)

    plt.title("Llama 3 Fine-Tuning Loss Curve", fontsize=16, fontweight='bold')
    plt.xlabel("Training Steps", fontsize=12)
    plt.ylabel("Loss (Lower is Better)", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()
else:
    print("⚠️ Trainer history not found. Did you restart the session? (Skip this plot if so)")

⚠️ Trainer history not found. Did you restart the session? (Skip this plot if so)


# Retrieval Performance Matrix (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup a Test Set (Random 500 rows)
test_df = df.sample(500, random_state=42)

# Focus on Top 8 Cuisines to keep the chart readable
top_cuisines = df['Cuisine'].value_counts().head(8).index.tolist()
test_df = test_df[test_df['Cuisine'].isin(top_cuisines)]

y_true = []
y_pred = []

print("⏳ Running evaluation on test set (this takes ~30 secs)...")

# 2. Run Semantic Search for each row
# We hide the 'Cuisine' from the query to make it a fair test
for index, row in test_df.iterrows():
    # Query: Ingredients only
    query = f"I have {row['Cleaned-Ingredients']}"
    query_vec = embedder.encode([query])

    # Search
    sims = embedder.similarity(query_vec, recipe_embeddings)[0]

    # Get Top 1 match (excluding itself implies robust retrieval,
    # but here we just check if top match aligns with cuisine category)
    top_idx = np.argmax(sims.numpy())
    pred_cuisine = df.iloc[top_idx.item()]['Cuisine']

    y_true.append(row['Cuisine'])
    y_pred.append(pred_cuisine)

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=top_cuisines)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_cuisines, yticklabels=top_cuisines)

plt.title("Recommendation System Performance Matrix\n(Did it guess the right cuisine based on ingredients?)", fontsize=14)
plt.ylabel("Actual Cuisine")
plt.xlabel("Recommended Cuisine")
plt.xticks(rotation=45, ha='right')
plt.show()

# Semantic Cluster

In [ ]:
from sklearn.manifold import TSNE

# 1. Select a subset to speed up plotting (2000 points)
subset_idx = np.random.choice(len(df), 2000, replace=False)
subset_embeddings = recipe_embeddings[subset_idx]
subset_cuisines = df.iloc[subset_idx]['Cuisine'].values

# Keep only top 6 cuisines for cleaner colors
top_6 = df['Cuisine'].value_counts().head(6).index
mask = np.isin(subset_cuisines, top_6)
subset_embeddings = subset_embeddings[mask]
subset_cuisines = subset_cuisines[mask]

# 2. Run t-SNE (Dimensionality Reduction)
print("⏳ Calculating t-SNE clusters (this creates the map)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_results = tsne.fit_transform(subset_embeddings.numpy())

# 3. Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=tsne_results[:,0], y=tsne_results[:,1],
    hue=subset_cuisines,
    palette='tab10',
    s=60, alpha=0.8
)
plt.title("Semantic Map of Recipes\n(Similar recipes naturally group together)", fontsize=16)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

# Ingredients cooccurance heat map

In [ ]:
from collections import Counter
import itertools

# 1. Count Co-occurrences of Top 15 Ingredients
all_ingredients = [x.split(',') for x in df['Cleaned-Ingredients'].str.lower().tolist()]
flat_ingredients = [item.strip() for sublist in all_ingredients for item in sublist if item]

# Get top 15 most common ingredients
top_ingredients = [x[0] for x in Counter(flat_ingredients).most_common(15)]

# Initialize Matrix
co_matrix = pd.DataFrame(0, index=top_ingredients, columns=top_ingredients)

# Fill Matrix
for ingredients in all_ingredients:
    # Filter only top ingredients present in this recipe
    present = [i.strip() for i in ingredients if i.strip() in top_ingredients]
    # Add count for every pair
    for i1, i2 in itertools.combinations(present, 2):
        co_matrix.at[i1, i2] += 1
        co_matrix.at[i2, i1] += 1

# 2. Plot Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(co_matrix, cmap="YlOrRd", annot=False, linewidths=.5)
plt.title("Ingredient Compatibility Heatmap\n(Darker = Often cooked together)", fontsize=15)
plt.show()

In [ ]:
!pip install evaluate rouge_score jiwer

import time
import torch
import numpy as np
import pandas as pd
import evaluate
from tqdm import tqdm

# Load Evaluators
bleu = evaluate.load("bleu")
wer = evaluate.load("wer")

# Setup Data (Use a dedicated test set of 50 rows)
test_data = df.sample(50, random_state=42) # Keep it small for speed

def calculate_metrics(model, tokenizer, test_data):
    print("⏳ Starting Evaluation on 50 samples...")

    # Storage
    references = []
    predictions = []
    latencies = []
    accuracies = []

    # Enable Inference Mode
    FastLanguageModel.for_inference(model)

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
        # 1. Prepare Inputs
        ingredients = row['Cleaned-Ingredients']
        cuisine = row['Cuisine']
        ground_truth = row['TranslatedInstructions'] # We compare against real instructions

        prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I have these ingredients: {ingredients}. I want a {cuisine} dish.

### Response:
"""
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        # 2. Measure Latency & Generate
        start_time = time.time()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

        end_time = time.time()

        # 3. Process Output
        generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        recipe_body = generated_text.split("### Response:")[-1].strip()

        # Calculate Latency (ms per token)
        num_tokens = len(outputs[0])
        time_taken = end_time - start_time
        latencies.append((time_taken / num_tokens) * 1000) # ms/token

        # Store for BLEU/WER
        predictions.append(recipe_body)
        references.append(ground_truth)

        # 4. Calculate "Ingredient Accuracy"
        # (Did the model actually use the ingredients requested?)
        input_ings = [x.strip().lower() for x in ingredients.split(',')]
        found_count = 0
        for ing in input_ings:
            if ing in recipe_body.lower():
                found_count += 1

        # Accuracy = % of input ingredients found in output
        if len(input_ings) > 0:
            accuracies.append(found_count / len(input_ings))
        else:
            accuracies.append(1.0)

    # --- Compute Final Scores ---

    # BLEU
    bleu_score = bleu.compute(predictions=predictions, references=references)

    # WER
    wer_score = wer.compute(predictions=predictions, references=references)

    # Perplexity (Approximation using loss on test set)
    # We define a separate function or use trainer if available,
    # but for inference-only, we skip the heavy math and rely on qualitative scores.
    # A standard Llama 3 fine-tune usually hits Perplexity ~1.5 - 2.0

    print("\n" + "="*40)
    print("📊 MODEL PERFORMANCE REPORT")
    print("="*40)
    print(f"1. Avg Latency:      {np.mean(latencies):.2f} ms/token")
    print(f"2. BLEU Score:       {bleu_score['bleu']:.4f} (Target: > 0.15 for creative text)")
    print(f"3. Word Error Rate:  {wer_score:.4f} (Lower is better)")
    print(f"4. Ingredient Acc:   {np.mean(accuracies)*100:.1f}% (Constraint Satisfaction)")
    print("="*40)

# Run it
calculate_metrics(model, tokenizer, test_data)

## Overall Code

In [ ]:
# ==========================================
# PART 1: INSTALLATION & SETUP
# ==========================================
import os
print("⏳ Installing dependencies... (This takes ~2 mins)")

# Install Unsloth (Optimized Training) & Metrics Libraries
os.system('pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
os.system('pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes')
os.system('pip install sentence-transformers evaluate rouge_score jiwer')

import torch
import numpy as np
import pandas as pd
import time
import evaluate
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Disable Weights & Biases login prompt
os.environ["WANDB_DISABLED"] = "true"

print("✅ Installation Complete.")

# ==========================================
# PART 2: DATA PREPARATION
# ==========================================
# Load your uploaded CSV
csv_file = "Cleaned_Indian_Food_Dataset.csv"

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"❌ Please upload {csv_file} to Colab files first!")

df = pd.read_csv(csv_file)

# Create Instruction Format (The "Chain of Thought" Logic)
# Input: "I have X. I want Y." -> Output: Full Recipe
def format_instruction(row):
    return {
        "instruction": f"I have these ingredients: {row['Cleaned-Ingredients']}. I want a {row['Cuisine']} dish.",
        "input": "",
        "output": f"Here is a recipe for {row['TranslatedRecipeName']}.\n\nIngredients:\n{row['TranslatedIngredients']}\n\nInstructions:\n{row['TranslatedInstructions']}"
    }

# Convert to JSONL for Unsloth
data = df.apply(format_instruction, axis=1).tolist()
train_data, test_data = train_test_split(data, test_size=0.05, random_state=42) # Keep 5% for testing metrics

train_dataset = Dataset.from_list(train_data)
train_dataset.to_json("indian_food_train.jsonl")

print(f"✅ Data Prepared: {len(train_data)} training rows, {len(test_data)} testing rows.")

# ==========================================
# PART 3: FINE-TUNING LLAMA 3 (Unsloth)
# ==========================================
print("⏳ Loading Llama 3 (8B) in 4-bit...")

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Add LoRA Adapters (The "Brain Surgery")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Define Prompt Template
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files="indian_food_train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

print("🚀 Starting Training (60 Steps)...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Fast run. Increase to 300 for better results.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

# Capture Training Loss for Perplexity
final_train_loss = trainer_stats.training_loss

# ==========================================
# PART 4: SEMANTIC BRIDGE (RAG)
# ==========================================
print("⏳ Building Semantic Index (The 'Bridge')...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create Vector Database
df['semantic_text'] = df['TranslatedRecipeName'] + " " + df['Cleaned-Ingredients'] + " " + df['Cuisine']
recipe_embeddings = embedder.encode(df['semantic_text'].tolist(), show_progress_bar=True)

# ==========================================
# PART 5: METRICS EVALUATION
# ==========================================
print("⏳ Running Evaluation on Test Set (50 samples)...")

bleu = evaluate.load("bleu")
wer = evaluate.load("wer")

FastLanguageModel.for_inference(model) # Enable fast inference

references = []
predictions = []
latencies = []
accuracies = []

# Select 50 random test cases
test_subset = pd.DataFrame(test_data).sample(50, random_state=42)

for idx, row in tqdm(test_subset.iterrows(), total=len(test_subset)):
    # Extract "Ground Truth"
    truth_text = row['output'].split("Instructions:\n")[-1] # We compare the instruction body

    # Generate Prediction
    prompt = alpaca_prompt.format(row['instruction'], "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()

    # Process
    gen_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    pred_body = gen_text.split("### Response:")[-1].strip()

    # Latency (ms/token)
    num_tokens = len(outputs[0])
    latencies.append(((end_time - start_time) / num_tokens) * 1000)

    # Store text
    predictions.append(pred_body)
    references.append(truth_text)

    # Ingredient Accuracy (Constraint Check)
    # Extract ingredients from the prompt instruction
    req_ingredients = row['instruction'].split("ingredients: ")[1].split(". I want")[0].lower().split(',')
    hit = 0
    for ing in req_ingredients:
        if ing.strip() in pred_body.lower():
            hit += 1
    accuracies.append(hit / len(req_ingredients) if len(req_ingredients) > 0 else 1.0)

# Compute Scores
bleu_score = bleu.compute(predictions=predictions, references=references)
wer_score = wer.compute(predictions=predictions, references=references)
perplexity = np.exp(final_train_loss)

# ==========================================
# PART 6: FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("🥗 INDIAN CHEF AI: PERFORMANCE REPORT 🥗")
print("="*50)
print(f"1. Perplexity (PPL):     {perplexity:.2f} (Lower is better)")
print(f"2. BLEU Score:           {bleu_score['bleu']:.4f} (Creativity Match)")
print(f"3. Word Error Rate:      {wer_score:.4f}")
print(f"4. Avg Response Latency: {np.mean(latencies):.2f} ms/token")
print(f"5. Ingredient Accuracy:  {np.mean(accuracies)*100:.1f}% (Constraint Satisfaction)")
print("="*50)
print("✅ System Ready.")

⏳ Installing dependencies... (This takes ~2 mins)


/tmp/ipython-input-4092191470.py:17: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/unsloth/models/rl_replacements.py:946: UserWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  import trl.experimental.openenv.utils as openenv_utils


✅ Installation Complete.


...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/186 [00:00<?, ?it/s]

⏳ Running Evaluation on Test Set (50 samples)...


100%|██████████| 50/50 [15:52<00:00, 19.05s/it]



🥗 INDIAN CHEF AI: PERFORMANCE REPORT 🥗
1. Perplexity (PPL):     3.92 (Lower is better)
2. BLEU Score:           0.0571 (Creativity Match)
3. Word Error Rate:      0.9733
4. Avg Response Latency: 55.43 ms/token
5. Ingredient Accuracy:  87.2% (Constraint Satisfaction)
✅ System Ready.


In [ ]:
# Core
! pip install -q gradio pandas numpy scikit-learn datasets evaluate sentence-transformers

# Unsloth + training stack
! pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
! pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes rouge_score jiwer


In [ ]:
import os
import time
import torch
import gradio as gr
import numpy as np
import pandas as pd

from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

os.environ["WANDB_DISABLED"] = "true"

# ---------------------------
# GLOBAL STATE
# ---------------------------
model = None
tokenizer = None
embedder = SentenceTransformer("all-MiniLM-L6-v2")
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# ---------------------------
# DATA PREP
# ---------------------------
def prepare_data(csv_file):
    df = pd.read_csv(csv_file)

    def format_instruction(row):
        return {
            "instruction": f"I have these ingredients: {row['Cleaned-Ingredients']}. I want a {row['Cuisine']} dish.",
            "input": "",
            "output": (
                f"Here is a recipe for {row['TranslatedRecipeName']}.\n\n"
                f"Ingredients:\n{row['TranslatedIngredients']}\n\n"
                f"Instructions:\n{row['TranslatedInstructions']}"
            )
        }

    data = df.apply(format_instruction, axis=1).tolist()
    train_data, _ = train_test_split(data, test_size=0.05, random_state=42)

    Dataset.from_list(train_data).to_json("train.jsonl")
    return f"✅ Dataset ready with {len(train_data)} samples."

# ---------------------------
# TRAINING
# ---------------------------
def train_model():
    global model, tokenizer

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/llama-3-8b-bnb-4bit",
        max_seq_length=2048,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        use_gradient_checkpointing="unsloth"
    )

    dataset = load_dataset("json", data_files="train.jsonl", split="train")

    def format_prompts(examples):
        texts = []
        for i, o in zip(examples["instruction"], examples["output"]):
            texts.append(alpaca_prompt.format(i, o) + tokenizer.eos_token)
        return {"text": texts}

    dataset = dataset.map(format_prompts, batched=True)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=60,
            learning_rate=2e-4,
            logging_steps=1,
            output_dir="outputs",
            optim="adamw_8bit"
        ),
    )

    trainer.train()
    FastLanguageModel.for_inference(model)
    return "🚀 Training complete. Model ready."

# ---------------------------
# INFERENCE
# ---------------------------
def generate_recipe(ingredients, cuisine):
    if model is None:
        return "❌ Train the model first."

    prompt = alpaca_prompt.format(
        f"I have these ingredients: {ingredients}. I want a {cuisine} dish.",
        ""
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

# ---------------------------
# GRADIO UI
# ---------------------------
with gr.Blocks(title="🥗 Indian Chef AI") as app:
    gr.Markdown("## 🥗 Indian Chef AI – Personalized Recipe Generator")

    with gr.Tab("1️⃣ Upload Dataset"):
        csv_file = gr.File(label="Upload Cleaned_Indian_Food_Dataset.csv")
        prep_btn = gr.Button("Prepare Dataset")
        prep_out = gr.Textbox()
        prep_btn.click(prepare_data, csv_file, prep_out)

    with gr.Tab("2️⃣ Train Model"):
        train_btn = gr.Button("Start Fine-Tuning (≈10 mins)")
        train_out = gr.Textbox()
        train_btn.click(train_model, outputs=train_out)

    with gr.Tab("3️⃣ Generate Recipe"):
        ing = gr.Textbox(label="Ingredients (comma separated)")
        cuisine = gr.Textbox(label="Cuisine Type")
        gen_btn = gr.Button("Generate Recipe")
        recipe_out = gr.Textbox(lines=12)
        gen_btn.click(generate_recipe, [ing, cuisine], recipe_out)

app.launch(debug=True)


In [ ]:
!pip install gradio pandas faiss-cpu sentence-transformers google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.1 MB/s eta 0:00:00


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl accelerate bitsandbytes transformers gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.6/179.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.0 MB/s eta 0:00:00


In [ ]:
!pip install gradio pandas faiss-cpu sentence-transformers google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 👨‍🍳 Hybrid AI Chef – FAISS + Gemini 2.5 Flash (Gradio)
# ============================================================

import os
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import textwrap
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from google.api_core import exceptions

# ------------------------------------------------------------
# 1. Utilities
# ------------------------------------------------------------

def md(text):
    return textwrap.indent(text.replace("•", "-"), "> ")

# ------------------------------------------------------------
# 2. Load Dataset
# ------------------------------------------------------------

df = pd.read_csv("Cleaned_Indian_Food_Dataset.csv")

df.dropna(
    subset=[
        "TranslatedRecipeName",
        "Cleaned-Ingredients",
        "TranslatedInstructions",
        "TotalTimeInMins",
    ],
    inplace=True,
)

df.reset_index(drop=True, inplace=True)

df["search_text"] = (
    df["TranslatedRecipeName"] + " " + df["Cleaned-Ingredients"]
)

# ------------------------------------------------------------
# 3. Build FAISS Semantic Search
# ------------------------------------------------------------

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(df["search_text"].tolist(), show_progress_bar=True)

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))

LAST_RESULTS = None
GEMINI_MODEL = None

# ------------------------------------------------------------
# 4. Gemini Loader (MANUAL API KEY)
# ------------------------------------------------------------

def load_gemini(api_key):
    global GEMINI_MODEL

    if not api_key.strip():
        GEMINI_MODEL = None
        return "❌ API key missing"

    try:
        genai.configure(api_key=api_key.strip())
        model = genai.GenerativeModel("models/gemini-2.5-flash")

        # quick test
        model.generate_content("Hello", request_options={"timeout": 20})

        GEMINI_MODEL = model
        return "✅ Gemini 2.5 Flash loaded successfully"

    except Exception as e:
        GEMINI_MODEL = None
        return f"⚠️ Gemini failed – running FAISS only\n\n{e}"

# ------------------------------------------------------------
# 5. Hybrid Recommendation
# ------------------------------------------------------------

def recommend(query):
    global LAST_RESULTS

    if not query.strip():
        return "⚠️ Enter a query"

    q_emb = embedder.encode([query])
    faiss.normalize_L2(q_emb)

    scores, idx = index.search(q_emb.astype("float32"), 5)
    results = df.iloc[idx[0]].copy()
    results["score"] = scores[0]

    LAST_RESULTS = results

    # -------- Gemini Re-ranking --------
    if GEMINI_MODEL:
        try:
            recipe_block = ""
            for _, r in results.iterrows():
                recipe_block += (
                    f"- {r['TranslatedRecipeName']} "
                    f"(Time: {int(r['TotalTimeInMins'])} mins) | "
                    f"{r['Cleaned-Ingredients']}\n"
                )

            prompt = f"""
User request: "{query}"

Pick the best 3 recipes from below.
If the request implies easy or quick food, prioritize fewer ingredients and lower cooking time.

RECIPES:
{recipe_block}
"""

            response = GEMINI_MODEL.generate_content(
                prompt,
                request_options={"timeout": 30},
            )

            return md(response.text)

        except exceptions.ResourceExhausted:
            pass
        except Exception:
            pass

    # -------- FAISS-only fallback --------
    text = "🤖 **Top semantic matches:**\n\n"
    for i, r in results.iterrows():
        text += (
            f"**{r['TranslatedRecipeName']}** "
            f"({int(r['TotalTimeInMins'])} mins)\n"
            f"*Ingredients:* {r['Cleaned-Ingredients']}\n\n"
        )

    return md(text)

# ------------------------------------------------------------
# 6. Get Instructions
# ------------------------------------------------------------

def instructions(name):
    if LAST_RESULTS is None:
        return "⚠️ Search first"

    row = LAST_RESULTS[
        LAST_RESULTS["TranslatedRecipeName"]
        .str.contains(name, case=False, regex=False)
    ]

    if row.empty:
        return "❌ Recipe not found"

    steps = row.iloc[0]["TranslatedInstructions"]

    if GEMINI_MODEL:
        try:
            response = GEMINI_MODEL.generate_content(
                f"Simplify these cooking steps:\n{steps}",
                request_options={"timeout": 30},
            )
            return md(response.text)
        except Exception:
            pass

    return md(f"**Original Instructions:**\n\n{steps}")

# ------------------------------------------------------------
# 7. Gradio UI
# ------------------------------------------------------------

with gr.Blocks(title="Hybrid AI Chef") as demo:
    gr.Markdown("# 👨‍🍳 Hybrid AI Chef (FAISS + Gemini 2.5 Flash)")
    gr.Markdown(
        "Semantic recipe recommendation using vector search.\n"
        "Gemini is used **only for reasoning and re-ranking**."
    )

    api_key = gr.Textbox(
        label="Gemini API Key",
        type="password",
        placeholder="Paste your API key here",
    )
    load_btn = gr.Button("Load Gemini")
    status = gr.Markdown()

    load_btn.click(load_gemini, api_key, status)

    gr.Markdown("---")

    query = gr.Textbox(
        label="What do you want to cook?",
        placeholder="e.g. an easy dish with chicken",
    )
    search_btn = gr.Button("Find Recipes")
    output = gr.Markdown()

    search_btn.click(recommend, query, output)

    gr.Markdown("---")

    recipe_name = gr.Textbox(
        label="Recipe name for instructions",
        placeholder="Type one recipe name",
    )
    instr_btn = gr.Button("Get Instructions")
    instr_out = gr.Markdown()

    instr_btn.click(instructions, recipe_name, instr_out)

demo.launch()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/186 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a228eefd77c136df90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
